In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go

snp = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CSPX ETF Stock Price History.csv")
china = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CNYA ETF Stock Price History.csv")
em = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "EIMI ETF Stock Price History.csv")
gold = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XAD5 ETF Stock Price History.csv")
india = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "INR ETF Stock Price History.csv")
mscieurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XMEU ETF Stock Price History.csv")
smallcapeurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "SXRJ ETF Stock Price History.csv")
ussmallcap = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CUSS ETF Stock Price History.csv")
silver = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "SSLN ETF Stock Price History.csv")

dfs = [snp, china, em, gold, india, mscieurope, smallcapeurope, ussmallcap, silver]

N = len(dfs)
T = len(dfs[0]) - 1

dfs = [df[["Date", "Price", "Vol."]] for df in dfs]

# Convert Date to datetime
dfs_new = []
for df in dfs:
    df["Date"] = pd.to_datetime(df["Date"])
    dfs_new.append(df)

dfs = dfs_new

names = ["snp", "china", "em", "gold", "india", "mscieurope",
         "smallcapeurope", "ussmallcap", "silver"]

dfs_renamed = []

for name, df in zip(names, dfs):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    
    df = df.rename(columns={
        "Price": f"Price_{name}",
        "Vol.": f"Vol_{name}"
    })
    
    dfs_renamed.append(df)

dfs = dfs_renamed

# Merge all on Date
df_merged = dfs[0]
for df in dfs[1:]:
    df_merged = df_merged.merge(df, on="Date", how="inner", suffixes=("", "_x"))

# Sort by date
df_merged = df_merged.sort_values("Date")

def correct_volume(vol_value):
    if isinstance(vol_value, (float, int)):
        return vol_value
    if vol_value[-1] == "K":
        return 1000 * float(vol_value[:-1])
    if vol_value[-1] == "M":
        return 1_000_000 * float(vol_value[:-1])
    return float(vol_value)

df_merged = df_merged.set_index("Date")

price_cols = [col for col in df_merged.columns if col.startswith("Price_")]
vol_cols = [col for col in df_merged.columns if col.startswith("Vol_")]

prices_df = df_merged[price_cols].apply(pd.to_numeric, errors="coerce")
volumes_df = df_merged[vol_cols].copy()

for col in volumes_df.columns:
    volumes_df[col] = volumes_df[col].apply(correct_volume)

volumes_df = volumes_df.fillna(0.0)

prices = prices_df.values  # shape: (T+1, N)
volumes = volumes_df.values  # shape: (T+1, N)

returns = prices[1:] / prices[:-1] - 1
prices = prices[1:]
volumes = volumes[1:]         # (T, N)
dates = df_merged.index[1:]       # (T,)
print(returns.shape, prices.shape, volumes.shape, dates.shape)

(2413, 9) (2413, 9) (2413, 9) (2413,)


In [2]:
graph = go.Figure()
for i in range(returns.shape[1]):
    L = list(range(T))
    cum_returns = np.cumprod(1 + returns[:,i]) - 1
    graph.add_trace(go.Scatter(x=dates, y=cum_returns, name=names[i]))
graph.show()

In [3]:
# ETFs spreads (bps) - estimated
etf_spreads = np.array([0.0075, 0.050, 0.100, 0.035, 0.130, 0.030, 0.030, 0.050, 0.08]) / 100
risk_free = 0.03
alpha = 0.05   # tail probability for CVaR
lambda_ = 0.5  # CVaR penalty
gamma = 0.08
weight_diff_rebalance = 0.10 # minimum weight difference for a portfolio rebalance
window_size = 252
validation_size = 252
m_params = np.array([0.33, 0.33, 0.33])
stress_corr_weight = 0.5
max_cash_alloc = 0.3
cash_alloc_param = 0.5
min_weight = 0.1
stepsize = 50

In [4]:
from rollingwindow_and_eval import rolling_window

df, total_costs, total_stress_values, track_pct_rebalances, track_cash_allocs = rolling_window(
    returns, volumes, prices,
     window_size, stepsize, validation_size,
     gamma, alpha, lambda_,
    m_params, risk_free, etf_spreads, weight_diff_rebalance,
    stress_corr_weight, max_cash_alloc, cash_alloc_param, min_weight
    )

df

C:\Users\tobia\AppData\Roaming\Python\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning:

Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.

C:\Users\tobia\AppData\Roaming\Python\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning:

Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.

C:\Users\tobia\AppData\Roaming\Python\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning:

Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please update the gencode to avoid c

DISTANCE:
[[0.00000000e+00 5.17512502e-01 3.00324379e-01 7.53067446e-01
  3.76657504e-01 2.95801297e-01 3.11543006e-01 1.88633114e-01
  7.05768429e-01]
 [5.17512502e-01 0.00000000e+00 4.64452259e-01 6.97863948e-01
  5.37388043e-01 5.42838204e-01 5.48180319e-01 5.41251175e-01
  6.52216586e-01]
 [3.00324379e-01 4.64452259e-01 0.00000000e+00 7.30760055e-01
  3.67681921e-01 3.37853414e-01 3.48014181e-01 3.24677398e-01
  6.74229621e-01]
 [7.53067446e-01 6.97863948e-01 7.30760055e-01 0.00000000e+00
  6.96541780e-01 7.71943920e-01 7.95749914e-01 7.54093472e-01
  3.89184262e-01]
 [3.76657504e-01 5.37388043e-01 3.67681921e-01 6.96541780e-01
  0.00000000e+00 4.13881981e-01 3.84046984e-01 4.23499347e-01
  7.08882434e-01]
 [2.95801297e-01 5.42838204e-01 3.37853414e-01 7.71943920e-01
  4.13881981e-01 0.00000000e+00 2.80266097e-01 3.30765334e-01
  6.86978699e-01]
 [3.11543006e-01 5.48180319e-01 3.48014181e-01 7.95749914e-01
  3.84046984e-01 2.80266097e-01 7.45058060e-09 3.33761649e-01
  7.71291823e-

c:\Projects\portfolio\further_opts\objectives.py:29: ClusterWarning:

The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix



DISTANCE:
[[0.00000000e+00 5.64723465e-01 3.52466785e-01 7.83825743e-01
  4.44833844e-01 3.32214907e-01 3.12170928e-01 2.21777166e-01
  6.90257122e-01]
 [5.64723465e-01 7.45058060e-09 4.07632729e-01 7.30734348e-01
  6.13651136e-01 5.59903551e-01 5.60187352e-01 5.27737810e-01
  6.71708364e-01]
 [3.52466785e-01 4.07632729e-01 0.00000000e+00 7.74141507e-01
  4.28320657e-01 3.69601274e-01 3.39920536e-01 3.28341121e-01
  6.85338120e-01]
 [7.83825743e-01 7.30734348e-01 7.74141507e-01 0.00000000e+00
  7.65102121e-01 8.12236729e-01 8.05120891e-01 7.60480908e-01
  3.78255905e-01]
 [4.44833844e-01 6.13651136e-01 4.28320657e-01 7.65102121e-01
  0.00000000e+00 4.38189708e-01 4.31860355e-01 4.70362038e-01
  7.28222914e-01]
 [3.32214907e-01 5.59903551e-01 3.69601274e-01 8.12236729e-01
  4.38189708e-01 0.00000000e+00 2.20881108e-01 3.57931665e-01
  7.09114245e-01]
 [3.12170928e-01 5.60187352e-01 3.39920536e-01 8.05120891e-01
  4.31860355e-01 2.20881108e-01 0.00000000e+00 3.21666581e-01
  7.15515008e-

,var_alpha,cvar,sharpe,sortino,mean_return,md
ER,-0.007370,-0.012318,0.835452,1.158390,0.000427,-0.130364
ER_cvar,-0.007378,-0.012330,0.835440,1.158381,0.000428,-0.130376
sharpe,-0.007283,-0.012521,0.850173,1.137961,0.000397,-0.130632
sharpe_cvar,-0.007282,-0.012520,0.850376,1.138222,0.000397,-0.130623
momentum_based,-0.007395,-0.012294,0.886040,1.209286,0.000351,-0.130657
momentum_cvar,-0.007395,-0.012295,0.885990,1.209387,0.000351,-0.130661
risk_parity,-0.007451,-0.012397,0.849843,1.175785,0.000441,-0.129830
hrp_weights,-0.007345,-0.012332,0.836150,1.159175,0.000428,-0.130332
equal_weights,-0.007367,-0.012314,0.835541,1.158505,0.000427,-0.130352


In [5]:
num_strats = 9
from plotly.subplots import make_subplots
fig = make_subplots(rows=1, cols=2)
L = list(range(len(total_costs[0])))
for i in range(num_strats):
    fig.add_trace(go.Scatter(x=L, y=total_costs[i], name=f"cost - {df.index[i]}"), row=1, col=1)
    fig.add_trace(go.Scatter(x=L, y=total_stress_values[i], name=f"stress - {df.index[i]}"), row=1, col=2)
fig.show()

In [6]:
num_strats = 9

fig = make_subplots(rows=1, cols=1)
L = list(range(len(total_costs[0])))
for i in range(num_strats):
    fig.add_trace(go.Scatter(x=L, y=track_cash_allocs[i], name=df.index[i]), row=1, col=1)
fig.show()

In [5]:
from stress_testing import stress_test
from rollingwindow_and_eval import compute_statistics_rolling

start_train = "2019-10-01"
end_train = "2020-02-01"
end_valid = "2020-03-01"

window_size = 252

all_corrs, all_volas = compute_statistics_rolling(returns, window_size, window_size)
global_corr_mean = np.mean(all_corrs)
global_corr_std = np.std(all_corrs)
global_vola_mean = np.mean(all_volas)
global_vola_std = np.std(all_volas)

Sigma = np.cov(returns, rowvar=False)

df_merged

# stats, cumreturns_all, ws, stress_values = stress_test(
#     df_merged, start_train, end_train, end_valid,
#     gamma, alpha, lambda_, m_params, Sigma,
#     min_weight, stress_corr_weight, global_corr_mean,
#     global_corr_std, global_vola_mean,
#     global_vola_std, max_cash_alloc,
#     cash_alloc_param, risk_free)

# stats



,Price_snp,Vol_snp,Price_china,Vol_china,Price_em,Vol_em,Price_gold,Vol_gold,Price_india,Vol_india,Price_mscieurope,Vol_mscieurope,Price_smallcapeurope,Vol_smallcapeurope,Price_ussmallcap,Vol_ussmallcap,Price_silver,Vol_silver
Date,,,,,,,,,,,,,,,,,,
2015-06-04,193.48,79.92K,6.29,3.59K,24.41,29.05K,102.72,5.42K,14.10,410.18K,3966.5,0.55K,157.48,0.12K,251.28,0.83K,1036.25,2.48K
2015-06-05,192.96,268.78K,6.32,NaN,24.23,1.37K,103.72,11.29K,14.26,140.22K,3906.0,52.00K,155.25,2.35K,250.14,NaN,1039.50,8.35K
2015-06-08,192.07,96.24K,6.48,30.00K,24.16,6.35K,102.95,3.62K,13.93,126.13K,3896.0,NaN,153.55,2.50K,250.78,6.15K,1028.75,0.47K
2015-06-09,191.68,114.16K,6.30,37.71K,24.07,15.51K,103.01,12.21K,13.84,384.04K,3876.0,0.13K,153.47,0.07K,249.49,NaN,1026.75,1.28K
2015-06-10,193.84,150.58K,6.41,0.36K,24.35,232.44K,103.60,1.38K,14.05,187.14K,3912.5,NaN,155.62,0.25K,252.62,11.49K,1016.50,0.41K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-16,717.87,67.65K,5.93,1.40M,47.39,412.44K,415.60,6.27K,24.95,64.88K,9404.0,9.04K,331.05,5.34K,599.70,2.92K,5702.00,441.38K
2026-03-17,721.96,80.25K,5.92,280.55K,47.99,265.78K,416.04,3.16K,24.95,14.36K,9465.0,4.24K,333.90,3.29K,604.70,2.04K,5669.00,438.25K
2026-03-18,716.48,132.07K,5.86,554.45K,47.58,982.99K,406.02,3.75K,24.66,44.02K,9399.0,7.31K,334.35,6.66K,602.00,3.01K,5501.00,391.98K
